# Reproduce Table 3 with pandas and statsmodels
This notebook loads `data/cleaned_data.csv`, reconstructs the same regression specifications used in `3-Analysis.do`, and presents the resulting Table 3 in pandas.

In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

pd.options.display.float_format = '{:.3f}'.format

data_path = 'data/cleaned_data.csv'
df = pd.read_csv(data_path, dtype={'code': str})
print(f'Loaded {len(df):,} rows from {data_path}')
df.head()

Loaded 13,818 rows from data/cleaned_data.csv


,id_in_session,code,index_in_pages,current_page_name,time_started,payoff,consent,treatment,age_temp,gender,...,mean_prob_rep_temp,mean_prob_rep,mean_prob_dem_temp,mean_prob_dem,mean_prob_net,mean_prob_net_mean,mean_prob_net_sd,mean_prob_net_z,out_of_range_q,out_of_range
0,213,00d576h7,155,Results,2018-06-25 20:15:41.255206+00:00,0,NaN,SND,NaN,NaN,...,NaN,0.517,NaN,0.700,-0.183,-0.064,0.184,-0.645,0,0
1,213,00d576h7,155,Results,2018-06-25 20:15:41.255206+00:00,0,NaN,SND,NaN,NaN,...,NaN,0.517,NaN,0.700,-0.183,-0.064,0.184,-0.645,0,0
2,213,00d576h7,155,Results,2018-06-25 20:15:41.255206+00:00,0,NaN,SND,NaN,NaN,...,NaN,0.517,0.700,0.700,-0.183,-0.064,0.184,-0.645,0,0
3,213,00d576h7,155,Results,2018-06-25 20:15:41.255206+00:00,0,NaN,SND,NaN,NaN,...,0.517,0.517,NaN,0.700,-0.183,-0.064,0.184,-0.645,0,0
4,213,00d576h7,155,Results,2018-06-25 20:15:41.255206+00:00,0,NaN,SND,NaN,NaN,...,NaN,0.517,NaN,0.700,-0.183,-0.064,0.184,-0.645,0,0


In [3]:
# Prepare analysis variables and data types for Table 3
bool_cols = [
    'pro_party', 'anti_party', 'true_news', 'fake_news', 'neutral_news',
    'message_greater', 'message_less', 'politicized_news', 'red_state',
    'religious_group', 'white', 'black', 'asian', 'latino', 'male',
    'polarizing'
]
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

# Use the same categorical fixed-effect variables as in the Stata code
for col in ['round_number', 'topic_id', 'code']:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Make sure the key outcome is numeric
if 'change_guess_message' in df.columns:
    df['change_guess_message'] = pd.to_numeric(df['change_guess_message'], errors='coerce')

# Sample selection used in Table 3
if 'pro_party' in df.columns and 'anti_party' in df.columns:
    df['only_political_news'] = (df['pro_party'] + df['anti_party']) == 1

# Show the relevant column types and a quick preview
cols_to_check = ['pro_party', 'anti_party', 'polarizing', 'prob_true', 'change_guess_message', 'round_number', 'topic_id', 'code']
print(df[cols_to_check].dtypes)
df[cols_to_check].head()

pro_party                  int64
anti_party                 int64
polarizing                 int64
prob_true                float64
change_guess_message     float64
round_number            category
topic_id                category
code                    category
dtype: object


C:\Users\Hex\AppData\Local\Temp\ipykernel_532\3643345521.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['only_political_news'] = (df['pro_party'] + df['anti_party']) == 1


,pro_party,anti_party,polarizing,prob_true,change_guess_message,round_number,topic_id,code
0,0,0,1,0.000,0.000,12,11,00d576h7
1,0,0,0,1.000,NaN,11,12,00d576h7
2,0,1,0,0.500,0.000,2,4,00d576h7
3,1,0,0,0.500,0.000,4,5,00d576h7
4,0,0,1,0.500,0.000,13,13,00d576h7


In [4]:
def run_regression(formula, data, cluster_col='code'):
    # Fit OLS and then re-fit the same sample with cluster-robust standard errors.
    ols_result = smf.ols(formula=formula, data=data).fit()
    row_labels = ols_result.model.data.row_labels
    used_data = data.loc[row_labels, :].copy()
    model = smf.ols(formula=formula, data=used_data).fit(
        cov_type='cluster', cov_kwds={'groups': used_data[cluster_col]}
    )
    return model

specs = [
    {
        'label': '(1)',
        'formula': 'change_guess_message ~ pro_party',
        'mask': df['only_political_news'],
    },
    {
        'label': '(2)',
        'formula': 'change_guess_message ~ polarizing',
        'mask': df['only_political_news'],
    },
    {
        'label': '(3)',
        'formula': 'change_guess_message ~ pro_party + polarizing',
        'mask': df['only_political_news'],
    },
    {
        'label': '(4)',
        'formula': 'change_guess_message ~ pro_party + prob_true',
        'mask': df['only_political_news'],
    },
    {
        'label': '(5)',
        'formula': 'change_guess_message ~ polarizing + prob_true',
        'mask': df['only_political_news'],
    },
    {
        'label': '(6)',
        'formula': 'change_guess_message ~ pro_party + polarizing + prob_true',
        'mask': df['only_political_news'],
    },
]

for spec in specs:
    data = df.loc[spec['mask'], :].copy()
    spec['model'] = run_regression(spec['formula'], data)
    spec['n'] = len(data)
    spec['mean'] = data['change_guess_message'].mean()

specs

[{'label': '(1)',
  'formula': 'change_guess_message ~ pro_party',
  'mask': 0        False
  1        False
  2         True
  3         True
  4        False
           ...  
  13813     True
  13814     True
  13815    False
  13816     True
  13817    False
  Name: only_political_news, Length: 13818, dtype: bool,
  'model': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x1d877391a90>,
  'n': 7902,
  'mean': np.float64(0.6582619339045288)},
 {'label': '(2)',
  'formula': 'change_guess_message ~ polarizing',
  'mask': 0        False
  1        False
  2         True
  3         True
  4        False
           ...  
  13813     True
  13814     True
  13815    False
  13816     True
  13817    False
  Name: only_political_news, Length: 13818, dtype: bool,
  'model': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x1d876f64050>,
  'n': 7902,
  'mean': np.float64(0.6582619339045288)},
 {'label': '(3)',
  'formula': 'change_guess_message ~ pro_party 

In [5]:
rows = [
    'Pro-Party News',
    'Pro-Party News SE',
    'Polarizing News',
    'Polarizing News SE',
    'P(True)',
    'P(True) SE',
    'Question FE',
    'Round FE',
    'Subject FE',
    'Observations',
    'R2',
    'Mean',
]

table_data = {spec['label']: [] for spec in specs}
for spec in specs:
    model = spec['model']
    table_data[spec['label']].extend([
        model.params.get('pro_party', np.nan),
        model.bse.get('pro_party', np.nan),
        model.params.get('polarizing', np.nan),
        model.bse.get('polarizing', np.nan),
        model.params.get('prob_true', np.nan),
        model.bse.get('prob_true', np.nan),
        'Yes',
        'Yes',
        'Yes',
        spec['n'],
        model.rsquared,
        spec['mean'],
    ])

results_display = pd.DataFrame(table_data, index=rows)
results_display

,(1),(2),(3),(4),(5),(6)
Pro-Party News,0.122,NaN,0.112,0.020,NaN,0.024
Pro-Party News SE,0.019,NaN,0.020,0.018,NaN,0.018
Polarizing News,NaN,0.068,0.041,NaN,-0.010,-0.015
Polarizing News SE,NaN,0.018,0.018,NaN,0.016,0.016
P(True),NaN,NaN,NaN,1.138,1.148,1.141
P(True) SE,NaN,NaN,NaN,0.055,0.054,0.056
Question FE,Yes,Yes,Yes,Yes,Yes,Yes
Round FE,Yes,Yes,Yes,Yes,Yes,Yes
Subject FE,Yes,Yes,Yes,Yes,Yes,Yes
Observations,7902,7902,7902,7902,7902,7902
